In [1]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg2zn11_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 39 atoms, Mg6Zn33


In [2]:
slab = surface(alloy, (2, 1, 0), 6, vacuum=8)
sc = make_supercell(slab, [[2,0,0],[0,2,0],[0,0,1]])

sym = np.array(sc.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = sc.get_positions()[:, 2]
thick = z.max() - z.min()
area = np.linalg.norm(np.cross(sc.cell[0], sc.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(2,1,0), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(sc)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")
print(f"Stoichiometric: {'YES' if mg*11 == zn*2 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 936, Mg: 144, Zn: 792
Thickness: 22.8 A
Area: 678.6 A²
Stoichiometric: YES
Symmetric: False


In [3]:
z = sc.get_positions()[:, 2]
sym = np.array(sc.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Tolerance: {tol:.3f} A")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

n = len(layers)
z_min, z_max = z.min(), z.max()
print(f"Total layers: {n}")

print("\nSearching for symmetric removals...\n")

found_any = False
for bot_rm in range(0, min(15, n//2)):
    for top_rm in range(0, min(15, n//2)):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) < 100:
            continue
        tr = sc[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(2,1,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                sym_tr = np.array(tr.get_chemical_symbols())
                mg_tr = sum(sym_tr == 'Mg')
                zn_tr = sum(sym_tr == 'Zn')
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"Mg{mg_tr} Zn{zn_tr}, removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
                found_any = True
        except:
            continue

if not found_any:
    print("No symmetric sub-slab found")

Tolerance: 0.062 A
Total layers: 108

Searching for symmetric removals...

  bot_rm=1, top_rm=0: 920 atoms, Mg144 Zn776, removed 16+0=16
  bot_rm=2, top_rm=1: 912 atoms, Mg144 Zn768, removed 20+4=24
  bot_rm=3, top_rm=2: 904 atoms, Mg144 Zn760, removed 24+8=32
  bot_rm=4, top_rm=3: 872 atoms, Mg144 Zn728, removed 40+24=64
  bot_rm=5, top_rm=4: 832 atoms, Mg144 Zn688, removed 60+44=104
  bot_rm=6, top_rm=5: 824 atoms, Mg136 Zn688, removed 64+48=112
  bot_rm=7, top_rm=6: 808 atoms, Mg136 Zn672, removed 72+56=128
  bot_rm=8, top_rm=7: 800 atoms, Mg128 Zn672, removed 76+60=136
  bot_rm=9, top_rm=8: 792 atoms, Mg128 Zn664, removed 80+64=144
  bot_rm=10, top_rm=9: 768 atoms, Mg112 Zn656, removed 92+76=168
  bot_rm=11, top_rm=10: 760 atoms, Mg112 Zn648, removed 96+80=176
  bot_rm=12, top_rm=11: 752 atoms, Mg104 Zn648, removed 100+84=184
  bot_rm=13, top_rm=12: 736 atoms, Mg104 Zn632, removed 108+92=200
  bot_rm=14, top_rm=13: 728 atoms, Mg96 Zn632, removed 112+96=208


In [4]:
removed_bot = layer_idx[0]
keep = []
for j in range(1, n):
    keep.extend(layer_idx[j])

trimmed = sc[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

rem_mg = sum(sym[j] == 'Mg' for j in removed_bot)
rem_zn = sum(sym[j] == 'Zn' for j in removed_bot)
print(f"\nRemoved: {len(removed_bot)} atoms (Mg{rem_mg} Zn{rem_zn})")

# Find mutual pairs among removed
ps_orig = AseAtomsAdaptor().get_structure(sc)
used = set()
mutual = []
for i, j1 in enumerate(removed_bot):
    if j1 in used:
        continue
    frac1 = ps_orig[j1].frac_coords
    inv1 = ps_orig.lattice.get_cartesian_coords((inv_translation - frac1) % 1.0)
    for j2 in removed_bot[i+1:]:
        if j2 in used:
            continue
        if np.linalg.norm(sc.positions[j2] - inv1) < 0.5:
            mutual.append((j1, j2))
            used.add(j1)
            used.add(j2)
            break

remaining = [j for j in removed_bot if j not in used]
print(f"Mutual pairs: {len(mutual)}")
print(f"Remaining unpaired: {len(remaining)}")

# Reconstruct mutual pairs
sc_fixed = sc.copy()
for keep_idx, move_idx in mutual:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac)

SG = P2/m
Inversion: f -> [0.8        0.         0.01374046] - f

Removed: 16 atoms (Mg0 Zn16)
Mutual pairs: 0
Remaining unpaired: 16


In [5]:
print("Unpaired atoms:")
print(f"{'idx':>5} {'el':>3} {'x':>8} {'y':>8} {'z':>8}  ->  {'inv_x':>8} {'inv_y':>8} {'inv_z':>8}")
print("-" * 75)

ps_orig = AseAtomsAdaptor().get_structure(sc)

for j in remaining:
    pos = sc.positions[j]
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    print(f"{j:>5} {sym[j]:>3} {pos[0]:>8.3f} {pos[1]:>8.3f} {pos[2]:>8.3f}  ->  "
          f"{inv_cart[0]:>8.3f} {inv_cart[1]:>8.3f} {inv_cart[2]:>8.3f}")

# Check which pairs map to each other
print("\nChecking mutual pairs:")
for i, j1 in enumerate(remaining):
    frac1 = ps_orig[j1].frac_coords
    inv1 = ps_orig.lattice.get_cartesian_coords((inv_translation - frac1) % 1.0)
    for j2 in remaining[i+1:]:
        d = np.linalg.norm(sc.positions[j2] - inv1)
        if d < 1.0:
            print(f"  {j1} <-> {j2}  (dist={d:.3f})")

Unpaired atoms:
  idx  el        x        y        z  ->     inv_x    inv_y    inv_z
---------------------------------------------------------------------------
  729  Zn    0.000   10.693    8.000  ->    31.164    6.728   31.373
  261  Zn   19.477   10.693    8.000  ->    11.686    6.728   31.373
  705  Zn   29.216   16.038    8.000  ->     1.948    1.383   31.373
  243  Zn   19.477   15.439    8.000  ->    11.686    1.982   31.373
    9  Zn   19.477    6.728    8.000  ->    11.686   10.693   31.373
  477  Zn    0.000    6.728    8.000  ->    31.164   10.693   31.373
  237  Zn    9.739   16.038    8.000  ->    21.425    1.383   31.373
   27  Zn   19.477    1.982    8.000  ->    11.686   15.439   31.373
  495  Zn    0.000    1.982    8.000  ->    31.164   15.439   31.373
  711  Zn    0.000   15.439    8.000  ->    31.164    1.982   31.373
    3  Zn    9.739    7.327    8.000  ->    21.425   10.094   31.373
  471  Zn   29.216    7.327    8.000  ->     1.948   10.094   31.373
  732  Zn  

In [6]:
# 8 mutual pairs identified from coordinates
move_pairs = [
    (27, 243),    # move 243 to inv(27)
    (9, 261),     # move 261 to inv(9)
    (495, 711),   # move 711 to inv(495)
    (477, 729),   # move 729 to inv(477)
    (30, 237),    # move 237 to inv(30)
    (3, 264),     # move 264 to inv(3)
    (498, 705),   # move 705 to inv(498)
    (471, 732),   # move 732 to inv(471)
]

sc_fixed = sc.copy()
ps_orig = AseAtomsAdaptor().get_structure(sc)

for keep_idx, move_idx in move_pairs:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    old = sc_fixed.positions[move_idx].copy()
    sc_fixed.positions[move_idx] = inv_cart
    print(f"Moved {move_idx} -> inv({keep_idx}): "
          f"({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
          f"({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(2,1,0), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if mg_f*11 == zn_f*2 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")

Moved 243 -> inv(27): (19.477, 15.439, 8.000) -> (11.686, 15.439, 31.373)
Moved 261 -> inv(9): (19.477, 10.693, 8.000) -> (11.686, 10.693, 31.373)
Moved 711 -> inv(495): (0.000, 15.439, 8.000) -> (31.164, 15.439, 31.373)
Moved 729 -> inv(477): (0.000, 10.693, 8.000) -> (31.164, 10.693, 31.373)
Moved 237 -> inv(30): (9.739, 16.038, 8.000) -> (21.425, 16.038, 31.373)
Moved 264 -> inv(3): (9.739, 10.094, 8.000) -> (21.425, 10.094, 31.373)
Moved 705 -> inv(498): (29.216, 16.038, 8.000) -> (1.948, 16.038, 31.373)
Moved 732 -> inv(471): (29.216, 10.094, 8.000) -> (1.948, 10.094, 31.373)

Atoms: 936, Mg: 144, Zn: 792
Stoichiometric: YES
Symmetric: True
Thickness: 23.4 A
Area: 678.6 A²


In [7]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [8]:
ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg2zn11_210_6layers_reconstructed_ase.data")

print(f"Saved: slabs/mg2zn11_210_6layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg2zn11_210_6layers_reconstructed_ase.data
  Atoms: 936
  Thickness: 23.4 A
  Area: 678.6 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [9]:
#unrelaxed surface energy calculation
E_slab = -998.83884       # eV
E_bulk_per_fu =  -44.119792 / 3  # eV per formula unit 
n = 936 / 13                 # formula units in slab = 32
A = 678.6            # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.706597 eV
n * E_bulk = -1058.87501 eV
E_slab - n*E_bulk = 60.03617 eV
S = 0.044235 eV/Ang^2
S = 0.7087 J/m^2


In [11]:
#relaxed surface energy calculation
E_slab_relaxed =  -1008.26413710508   # eV
E_bulk_per_fu = -44.119792 / 3  # eV per formula unit
n = 936 / 13                      # 32 formula units
A = 678.6                # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 =  0.7087
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 50.61087 eV
S relaxed = 0.037291 eV/Ang^2
S relaxed = 0.5975 J/m^2

Unrelaxed S = 0.7087 J/m^2
Relaxed S   = 0.5975 J/m^2
Relaxation energy = 0.1112 J/m^2
